In [16]:
import numpy as np

def sample_weights(num_samples, dim):
    """Sample random weight vectors from the feasible set Z_t.
    Here we assume Z_t is the unit simplex in `dim` dimensions (weights sum to 1, all nonnegative).
    This returns an array of shape (num_samples, dim) where each row is a weight vector."""
    # Use a Dirichlet(1,...,1) distribution to sample uniformly from the simplex
    return np.random.dirichlet(alpha=np.ones(dim), size=num_samples)

In [17]:
def generate_candidates(dim):
    """Generate candidate queries (Y, Y') that differ on specific frames.
    Here we generate queries comparing every pair of frames i and j:
    Y has 1.0 on frame i, Y' has 1.0 on frame j (all other entries 0)."""
    candidates = []
    for i in range(dim):
        for j in range(i+1, dim):
            Y = np.zeros(dim);  Y[i] = 1.0
            Y_prime = np.zeros(dim);  Y_prime[j] = 1.0
            candidates.append((Y, Y_prime))
    return candidates


In [18]:
def classify_query_outcomes(query, samples, tau):
    """Classify outcomes for a given query over all weight samples.
    Returns a tuple (pref_count, rev_count, indiff_count) indicating how many samples 
    would prefer Y, prefer Y', or be indifferent (within ±tau)."""
    Y, Y_prime = query
    # Compute margin for each sample weight: w · (Y - Y')
    margins = samples.dot(Y - Y_prime)
    # Count samples in each region
    pref_count   = np.sum(margins >  tau)
    rev_count    = np.sum(margins < -tau)
    indiff_count = np.sum(np.abs(margins) <= tau)
    return int(pref_count), int(rev_count), int(indiff_count)


In [19]:
def expected_volume_removed(pref_count, rev_count, indiff_count, total_samples):
    """Compute the expected volume removed by a query, given outcome sample counts.
    Volume is approximated by number of samples. Returns the expected number of samples removed."""
    N = total_samples
    # Estimate probability of each outcome
    pr_pref   = pref_count   / N
    pr_rev    = rev_count    / N
    pr_indiff = indiff_count / N
    # Expected remaining samples after query (weighted by outcome probabilities)
    expected_remaining = (pr_pref * pref_count) + (pr_rev * rev_count) + (pr_indiff * indiff_count)
    expected_removed = N - expected_remaining
    return expected_removed


In [ ]:
def score_query(query, samples, tau):
    """Calculate the expected volume removed by `query` on the given sample set."""
    pref_count, rev_count, indiff_count = classify_query_outcomes(query, samples, tau)
    return expected_volume_removed(pref_count, rev_count, indiff_count, len(samples))

def select_best_query(candidates, samples, tau):
    """Select the candidate query with the highest expected volume removal."""
    best_query = None
    best_score = -np.inf
    for query in candidates:
        score = score_query(query, samples, tau)
        if score > best_score:
            best_score = score
            best_query = query
    return best_query, best_score


In [21]:
def update_samples(samples, query, outcome, tau):
    """Update the sample set by removing weight vectors inconsistent with the observed outcome."""
    Y, Y_prime = query
    margins = samples.dot(Y - Y_prime)
    if outcome == 'pref':
        # Keep weights that would indeed prefer Y (margin > tau)
        new_samples = samples[margins > tau]
    elif outcome == 'rev':
        # Keep weights that prefer Y' (margin < -tau)
        new_samples = samples[margins < -tau]
    elif outcome == 'indiff':
        # Keep weights that yield indifference (|margin| <= tau)
        new_samples = samples[np.abs(margins) <= tau]
    else:
        raise ValueError("Unknown outcome type.")
    return new_samples


In [22]:
def simulate_user_response(query, true_weight, tau):
    """Simulate a user's response for a given query (Y, Y') and true weight vector.
    Returns 'pref' if true_weight · (Y - Y') > tau, 'rev' if < -tau, or 'indiff' otherwise."""
    Y, Y_prime = query
    true_margin = true_weight.dot(Y - Y_prime)
    if true_margin > tau:
        return 'pref'
    elif true_margin < -tau:
        return 'rev'
    else:
        return 'indiff'


In [23]:
# Example usage of the active learning loop with simulation
np.random.seed(42)  # for reproducibility

# Define problem dimensions and parameters
L = 5                                 # number of frames (attributes)
true_weight = np.random.dirichlet(np.ones(L))  # unknown true weight vector (in simplex)
tau = 0.1                             # indifference threshold

# Initialize sample set and candidate queries
samples = sample_weights(num_samples=5000, dim=L)
candidates = generate_candidates(dim=L)

print(f"True weight (hidden): {true_weight}\n")
# Active learning iterations
for t in range(1, 6):  # simulate 5 query iterations
    # Select the best query based on current samples
    query, score = select_best_query(candidates, samples, tau)
    # Simulate the user's response to the query
    outcome = simulate_user_response(query, true_weight, tau)
    # Identify which frames were compared (for logging)
    Y, Y_prime = query
    frame_diff = np.where(Y != Y_prime)[0]  # indices of frames that differ
    print(f"Iteration {t}: query comparing frames {frame_diff.tolist()}, " 
          f"expected removal ≈ {score:.1f} samples, user outcome = {outcome}")
    # Update the sample set based on the outcome
    samples = update_samples(samples, query, outcome, tau)
    print(f"  Remaining samples after update: {len(samples)}\n")
    if len(samples) == 0:
        print("Feasible region fully collapsed (no samples remain).")
        break


True weight (hidden): [0.07982511 0.51203839 0.22398576 0.1552966  0.02885413]

Iteration 1: query comparing frames [0, 3], expected removal ≈ 3333.2 samples, user outcome = indiff
  Remaining samples after update: 1668

Iteration 2: query comparing frames [1, 2], expected removal ≈ 1109.1 samples, user outcome = pref
  Remaining samples after update: 566

Iteration 3: query comparing frames [3, 4], expected removal ≈ 363.3 samples, user outcome = pref
  Remaining samples after update: 116

Iteration 4: query comparing frames [0, 1], expected removal ≈ 71.3 samples, user outcome = rev
  Remaining samples after update: 59

Iteration 5: query comparing frames [0, 2], expected removal ≈ 31.2 samples, user outcome = rev
  Remaining samples after update: 4

